# 00 · Data prep, design reconciliation & QC
Parses the **real** folder names, groups scans into **physical fruits**, and surfaces the leakage risk. The data is longitudinal: one fruit is µCT-scanned at ~4 days, so **one folder = one scan (fruit × day)**, not one fruit.

Layout: `data_root/Day{4,5,7,10}/<scan_folder>/vertical_section_*.jpg` (72 slices).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd
from config import CFG
import dataset as ds, eval_core as ec
CFG.out_dir.mkdir(parents=True, exist_ok=True)
print('data_root :', CFG.data_root); print('fruit_key :', CFG.fruit_key)

## Step 1 — index physical fruits

In [ ]:
fruits = ds.build_fruit_index(CFG.data_root, fruit_key=CFG.fruit_key)[0]
df = pd.DataFrame([{'fruit_id':f.fruit_id,'cohort':f.cohort,'dose':f.dose,
                    'rep':f.rep,'label':f.label,'n_days':len(f.scans),
                    'days':[s.day for s in f.scans]} for f in fruits])
df.head(10)

## Step 2 — reconciled design (answers the 48-vs-111 question)

In [ ]:
n_scan = sum(len(f.scans) for f in fruits)
print(f'physical fruits : {len(df)}  ({int(df.label.sum())} infested / {int((df.label==0).sum())} control)')
print(f'total scans     : {n_scan}')
print(f'images (approx)  : {n_scan} scans x {CFG.slices_per_fruit} slices = {n_scan*CFG.slices_per_fruit}')
print('\nfruits by cohort x dose:')
print(df.groupby(['cohort','dose']).size().unstack(fill_value=0))
print('\ndays-per-fruit:', dict(df.n_days.value_counts().sort_index()))
display(df[df.n_days<4][['fruit_id','days']])

> **Leakage note.** Because each physical fruit appears in several day-scans, the split in notebook 01 groups by `fruit_id` so a fruit's scans never straddle train/test. Splitting by scan or day would leak fruit identity across time.

## Step 3 — QC: every scan should have 72 slices

In [ ]:
rows=[]
for f in fruits:
    for s in f.scans:
        rows.append({'fruit_id':f.fruit_id,'day':s.day,
                     'n_slices':len(ds.slice_paths(s.directory))})
scans=pd.DataFrame(rows)
bad=scans[scans.n_slices!=CFG.slices_per_fruit]
print(f'scans with != {CFG.slices_per_fruit} slices: {len(bad)}'); display(bad.head())
scans.to_csv(CFG.out_dir/'scan_manifest.csv', index=False)

## Step 4 — view example slices (control vs infested)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
def show(fruit, k=8):
    paths = ds.slice_paths(fruit.scans[0].directory)[:k]
    fig,ax=plt.subplots(1,len(paths),figsize=(2*len(paths),2))
    for a,p in zip(np.atleast_1d(ax),paths): a.imshow(Image.open(p),cmap='gray'); a.axis('off')
    fig.suptitle(f"{fruit.fruit_id}  day{fruit.scans[0].day}  "
                 f"({'INFESTED' if fruit.label else 'CONTROL'})"); plt.show()
show(next(f for f in fruits if f.label==0)); show(next(f for f in fruits if f.label==1))

## Step 5 — save fruit manifest

In [ ]:
df.to_csv(CFG.out_dir/'fruit_manifest.csv', index=False)
print('saved fruit_manifest.csv + scan_manifest.csv')